### **Imports**

In [181]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

from sklearn.model_selection import KFold, train_test_split, RandomizedSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

### **Loading data**

In [182]:
df = pd.read_csv("used_cars.csv")
print(f"Shape: {df.shape}")

Shape: (4009, 12)


In [183]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [184]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         4009 non-null   object
 1   model         4009 non-null   object
 2   model_year    4009 non-null   int64 
 3   milage        4009 non-null   object
 4   fuel_type     3839 non-null   object
 5   engine        4009 non-null   object
 6   transmission  4009 non-null   object
 7   ext_col       4009 non-null   object
 8   int_col       4009 non-null   object
 9   accident      3896 non-null   object
 10  clean_title   3413 non-null   object
 11  price         4009 non-null   object
dtypes: int64(1), object(11)
memory usage: 376.0+ KB


### **Data cleaning**

In [185]:
df["price"] = df["price"].str.replace("$","").str.replace(",", "").astype(float)
df["milage"] = df["milage"].str.replace("mi.","").str.replace(" mi.", "").str.replace(",","").astype(float)
df["accident"] = df["accident"].fillna("Unknown")
df["had_accident"] = df["accident"].apply(lambda x: 1 if 'accident' in x.lower() else 0)
df["accident_unknown"] = df["accident"].apply(lambda x: 1 if x == 'Unknown' else 0)
df = df.drop(columns=['accident'])
df["clean_title"] = df["clean_title"].fillna("No")
df["clean_title"] = df["clean_title"].map({'Yes': 1, 'No': 0})
df["fuel_type"] = df["fuel_type"].fillna("Unknown")

df.isnull().sum()

brand               0
model               0
model_year          0
milage              0
fuel_type           0
engine              0
transmission        0
ext_col             0
int_col             0
clean_title         0
price               0
had_accident        0
accident_unknown    0
dtype: int64

### **Feature engineering**

In [186]:
CURRENT_YEAR = datetime.datetime.now().year
df["car_age"] = CURRENT_YEAR - df["model_year"]

df["horsepower"] = (df["engine"].str.extract(r"(\d+\.?\d*)HP", expand=False).astype(float))
df['horsepower_missing'] = df['horsepower'].isna().astype(int)
df["milage_per_year"] = df["milage"] / (df["car_age"]+1) # to avoid division by 0

luxury_brands = ['Lexus', 'INFINITI', 'Audi', 'Acura', 'BMW', 'Tesla', 'Land', 
    'Aston', 'Lincoln', 'Jaguar', 'Mercedes-Benz', 'Genesis', 'Bentley', 
    'Lucid', 'Porsche', 'Cadillac', 'Lamborghini', 
    'Maserati', 'Rivian', 'Alfa', 'Ferrari', 'Bugatti', 
    'Rolls-Royce', 'McLaren', 'Lotus', 'Karma', 'Maybach']
df["is_luxury"] = df["brand"].apply(lambda x: 1 if x in luxury_brands else 0)

# target encoding brand using K-Fold CV to prevent overfitting
def target_encode_brand(df, target_col="price", brand_col="brand", n_splits=5):
    encoded = np.zeros(len(df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(df):
        brand_means = df.iloc[train_idx].groupby(brand_col)[target_col].mean()
        encoded[val_idx] = df.iloc[val_idx][brand_col].map(brand_means)

    global_mean = df[target_col].mean()
    encoded = pd.Series(encoded, index=df.index).fillna(global_mean)

    return encoded

df['brand_encoded'] = target_encode_brand(df, target_col="price", brand_col="brand")
df = df.drop(columns=['brand'])

fuel_dummies = pd.get_dummies(df["fuel_type"], prefix="fuel", drop_first=True).astype(int)
df = pd.concat([df, fuel_dummies], axis=1)
df = df.drop(columns=['fuel_type'])

### **Outlier removal**

In [187]:
print(df["price"].describe())

count    4.009000e+03
mean     4.455319e+04
std      7.871064e+04
min      2.000000e+03
25%      1.720000e+04
50%      3.100000e+04
75%      4.999000e+04
max      2.954083e+06
Name: price, dtype: float64


In [188]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

print(f"IQR: ${IQR:,.0f}")
print(f"Q1: ${Q1:,.0f} | Q3: ${Q3:,.0f}")

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)].copy()

removed = len(df) - len(df_clean)
print(f"Removed {removed:,} outliers ({removed/len(df)*100:.1f}%)")
print(f"Old range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")
print(f"New range: ${df_clean['price'].min():,.0f} - ${df_clean['price'].max():,.0f}")

df = df_clean

IQR: $32,790
Q1: $17,200 | Q3: $49,990
Removed 244 outliers (6.1%)
Old range: $2,000 - $2,954,083
New range: $2,000 - $99,000


### **Train/test split**

In [ ]:
fuel_columns = [col for col in df.columns if col.startswith("fuel_")]

features = ["milage", "car_age", "horsepower", "horsepower_missing","had_accident", "accident_unknown", 
            "clean_title", "is_luxury", "brand_encoded", "milage_per_year"] + fuel_columns

X = df[features].copy()
y = df["price"].copy()

# target log transformation
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# for baseline comparison
_, _, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain price: ${y_train_raw.min():,.0f} - ${y_train_raw.max():,.0f} (mean: ${y_train_raw.mean():,.0f})")
print(f"Test price:  ${y_test_raw.min():,.0f} - ${y_test_raw.max():,.0f} (mean: ${y_test_raw.mean():,.0f})")

Train samples: 3012
Test samples: 753

Train price: $2,000 - $99,000 (mean: $33,531)
Test price:  $2,500 - $96,900 (mean: $33,467)


### **Missing values imputation**

In [190]:
print(f"Missing values in train set:")
train_missing = X_train.isna().sum()
if train_missing.sum() == 0:
    print("No missing values (all already handled)")
else:
    print(train_missing[train_missing > 0])

Missing values in train set:
horsepower    556
dtype: int64


In [191]:
train_medians = X_train.median()

print(f"Calculated medians from training data only:")
for col in features:
    if X_train[col].isna().sum() > 0:
        print(f"{col}: {train_medians[col]:.2f}")
    elif col == 'horsepower':
        print(f"{col}: {train_medians[col]:.2f} (calculated from train)")

X_train_filled = X_train.fillna(train_medians)
X_test_filled = X_test.fillna(train_medians)

print(f"Train missing after fill: {X_train_filled.isna().sum().sum()}")
print(f"Test missing after fill: {X_test_filled.isna().sum().sum()}")

Calculated medians from training data only:
horsepower: 305.00
Train missing after fill: 0
Test missing after fill: 0


### **Baseline model**

In [192]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train_filled, y_train_raw)
y_pred_baseline = baseline.predict(X_test_filled)

baseline_mae = mean_absolute_error(y_test_raw, y_pred_baseline)
baseline_rmse = root_mean_squared_error(y_test_raw, y_pred_baseline)
baseline_r2 = r2_score(y_test_raw, y_pred_baseline)

print(f"MAE: ${baseline_mae:.2f}")
print(f"RMSE: ${baseline_rmse:.2f}")
print(f"R^2: {baseline_r2:.3f}")

MAE: $16896.63
RMSE: $20936.13
R^2: -0.000


### **Random forest with hyperparameter tuning**

In [ ]:
param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_random = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_random.fit(X_train_filled, y_train_log)

print("Best parameters:")
for param, value in rf_random.best_params_.items():
    print(f"{param}: {value}")

best_cv_mae = -rf_random.best_score_
print(f"Best CV MAE (log scale): {best_cv_mae:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters:
n_estimators: 500
min_samples_split: 5
min_samples_leaf: 1
max_features: None
max_depth: None
Best CV MAE (log scale): 0.2355


In [ ]:
y_pred_log = rf_random.predict(X_test_filled)
y_pred = np.expm1(y_pred_log)
y_test = np.expm1(y_test_log)

rf_mae = mean_absolute_error(y_test, y_pred)
rf_rmse = root_mean_squared_error(y_test, y_pred)
rf_r2 = r2_score(y_test, y_pred)

print(f"MAE: ${rf_mae:,.2f} ({(1-rf_mae/baseline_mae)*100:+.1f}% vs baseline)")
print(f"RMSE: ${rf_rmse:,.2f}")
print(f"R^2: {rf_r2:.4f}")

MAE: $6,823.49 (+59.6% vs baseline)
RMSE: $10,018.55
R^2: 0.7710
